In [1]:
!mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.833 seconds
  Name    Version  Build                Channel
-------------------------------------------------------------
+ pandas  3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
- pip     26.2.2   pyh145f28c_0         conda-forge


In [6]:
import pandas as pd
import numpy as np

# Путь к исходному файлу
csv_path = "data/orders_2024_01.csv"

In [8]:
chunk_size = 50_000

total_revenue = 0
rows_count = 0

for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    # Рассчитываем выручку для каждой строки
    chunk["Revenue"] = chunk["price"] * chunk["quantity"]

    # Добавляем выручку текущего блока к общей
    total_revenue += chunk["Revenue"].sum()

    # Считаем количество строк
    rows_count += len(chunk)

print("Общая выручка:", total_revenue)
print("Количество строк:", rows_count)

Общая выручка: 129378775.5
Количество строк: 5000


In [9]:
use_columns = ["order_id", "user_id", "price", "quantity"]

data_types = {
    "order_id": "int32",
    "user_id": "int32",
    "price": "float32",
    "quantity": "int16"
}

df_optimized = pd.read_csv(
    csv_path,
    usecols=use_columns,
    dtype=data_types
)

# Проверяем структуру таблицы
print(df_optimized.head())

# Проверяем типы данных
print(df_optimized.dtypes)

# Получаем информацию о таблице
print(df_optimized.info())

   order_id  user_id         price  quantity
0         1     1044  14038.150391         1
1         2     1079   9172.599609         1
2         3     1041   9035.259766         2
3         4     1083  15234.330078         3
4         5     1068   4134.379883         2
order_id      int32
user_id       int32
price       float32
quantity      int16
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  5000 non-null   int32  
 1   user_id   5000 non-null   int32  
 2   price     5000 non-null   float32
 3   quantity  5000 non-null   int16  
dtypes: float32(1), int16(1), int32(2)
memory usage: 68.4 KB
None


In [10]:
# Загружаем исходный файл без дополнительных параметров
df_full = pd.read_csv(csv_path)

# Рассчитываем использование памяти
memory_full = df_full.memory_usage(deep=True).sum() / 1024**2
memory_optimized = df_optimized.memory_usage(deep=True).sum() / 1024**2

print(f"Память исходной таблицы: {memory_full:.2f} МБ")
print(f"Память оптимизированной таблицы: {memory_optimized:.2f} МБ")

# Рассчитываем экономию памяти
memory_saving = (1 - memory_optimized / memory_full) * 100

print(f"Экономия памяти: {memory_saving:.2f}%")

Память исходной таблицы: 0.54 МБ
Память оптимизированной таблицы: 0.07 МБ
Экономия памяти: 87.70%


In [11]:
user_revenue_parts = []

for chunk in pd.read_csv(
    csv_path,
    chunksize=chunk_size,
    usecols=use_columns,
    dtype=data_types
):
    # Рассчитываем выручку
    chunk["Revenue"] = chunk["price"] * chunk["quantity"]

    # Суммируем выручку по каждому пользователю в текущем блоке
    revenue_by_user = chunk.groupby("user_id")["Revenue"].sum()

    user_revenue_parts.append(revenue_by_user)

# Объединяем результаты всех блоков
user_revenue = pd.concat(user_revenue_parts).groupby(level=0).sum()

# Преобразуем Series в DataFrame
user_revenue = user_revenue.reset_index()

# Переименовываем столбец с выручкой
user_revenue.columns = ["user_id", "total_revenue"]

# Сохраняем результат в CSV
user_revenue.to_csv(
    "user_revenue_2024_01.csv",
    index=False
)

print(user_revenue.head())
print("Файл успешно сохранён!")

   user_id  total_revenue
0     1001   1.679781e+06
1     1002   7.561577e+05
2     1003   1.290829e+06
3     1004   1.373916e+06
4     1005   8.020001e+05
Файл успешно сохранён!
